# UAPP — Structure-aware backbone (SaProt) on T2837 + S669

**Question.** §11 showed that ESM2-650M's σ-branch ranking transferred from T2837 to S669 (Spearman 0.348 → 0.434), but μ-accuracy did not (RMSE 1.50 → 2.83).  The campaign so far swapped datasets twice but kept the encoder fixed.  Can a structure-aware backbone (**SaProt** = AA + 3Di tokens via FoldSeek) rescue μ on the new label distribution while preserving σ-ranking?

**Method.** Drop-in encoder swap.  We rebuild the embedding cache with SaProt for both T2837 and S669, keeping the rest of the pipeline (D5 head, ensemble training, σ recalibration) identical to §11.

**Why both datasets, not just S669.**
1. T2837 is the in-distribution baseline — without it we can't tell whether SaProt is genuinely improving or just shifting the basin.
2. The strict-improvement criterion needs both halves to hold:
   - T2837: Spearman ≥ 0.348 and RMSE ≤ 1.50  (don't break what works)
   - S669:  RMSE significantly < 2.83  (the actual hypothesis)
3. The σ recalibration scalar (T = 2.76 with ESM2) is encoder-specific; it must be re-fitted for SaProt on S669.

**Compute.** ESM2-650M took ~3 min on T4 to cache T2837 (2584 mutations across 108 proteins).  SaProt is the same parameter count and hidden size; expect the same wall-clock plus ~1 min for FoldSeek.  Total ~10 min for both datasets.

## 1. Environment

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf uapp
!git clone https://github.com/RoselindSi/uapp.git
%cd /content/uapp
!git checkout claude/saprot-backbone   # use this PR's branch until merged
!git log --oneline -3

In [ ]:
!pip install -q transformers torch numpy pandas tqdm scipy biopython scikit-learn
!apt-get install -y foldseek > /dev/null 2>&1 || (echo 'apt foldseek missing; trying conda' && pip install -q foldseek)
!foldseek version

## 2. Restore prerequisites from Drive

We need:
- T2837 metadata + ESM2 cache (for the split assignment and the σ-recalibration baseline)
- S669 metadata + ESM2 cache + bundled WT PDBs (already saved to Drive in the §11 run)
- T2837 AlphaFold-DB PDBs from scripts/14 (one per uniprot_id)

If T2837 AF PDBs are not in Drive, the next cell rebuilds them via scripts/14 (~5 min).

In [ ]:
import os, shutil
os.makedirs('cache', exist_ok=True)
os.makedirs('data/s669/S669', exist_ok=True)

to_restore = [
    ('cache/t2837_embeddings_v2_650m.pt',       'restore'),
    ('cache/t2837_metadata.csv',                'restore'),
    ('cache/t2837_bio_features_650m_extended.pt', 'restore'),
    ('cache/s669_metadata.csv',                 'restore'),
    ('cache/s669_metadata_processed.csv',       'restore'),
    ('cache/s669_embeddings_650m.pt',           'restore'),
    ('cache/s669_bio_features_650m_extended.pt','restore'),
]
for path, _ in to_restore:
    src = f'/content/drive/MyDrive/uapp_cache/{os.path.basename(path)}'
    if os.path.exists(src):
        shutil.copy(src, path)
        print(f'✓ {path}')
    else:
        print(f'✗ missing: {path}')

# S669 PDBs — re-download the Zenodo zip if not in Drive
if not os.path.exists('data/s669/S669/pdbs') or not os.listdir('data/s669/S669/pdbs'):
    print('\nFetching S669.zip from Zenodo (one-time)...')
    !wget -q -O data/s669/S669.zip https://zenodo.org/records/7568094/files/S669.zip
    !unzip -o -q data/s669/S669.zip -d data/s669/
print(f's669 pdbs: {len(os.listdir("data/s669/S669/pdbs"))}')

## 3. T2837 AlphaFold-DB PDBs

scripts/14 downloads one AF DB model per `uniprot_id` (~99 unique).  If the cache is in Drive, restore it; otherwise rebuild.

In [ ]:
import os
drive_af = '/content/drive/MyDrive/uapp_cache/af_pdbs'
local_af = 'af_pdbs'
script14_af = 'cache/af_pdbs'  # where script 14 actually writes

if os.path.exists(drive_af) and os.listdir(drive_af):
    !cp -r {drive_af} {local_af}
    print(f'restored {len(os.listdir(local_af))} AF PDBs from Drive')
elif os.path.exists(script14_af) and os.listdir(script14_af):
    if not os.path.exists(local_af):
        os.symlink(os.path.abspath(script14_af), local_af)
    print(f'using existing {len(os.listdir(local_af))} AF PDBs at {script14_af}')
else:
    print('AF PDBs not in Drive — rebuilding via scripts/14 (one-time, ~5 min)...')
    !python scripts/14_compute_structural_features.py \
        --metadata-csv cache/t2837_metadata.csv \
        --embeddings   cache/t2837_embeddings_v2_650m.pt \
        --extended-bio cache/t2837_bio_features_650m_extended.pt \
        --out          /tmp/t2837_dssp_features.pt
    # Script 14 caches PDBs at cache/af_pdbs — symlink to ./af_pdbs for the rest of the notebook
    if os.path.exists(script14_af) and not os.path.exists(local_af):
        os.symlink(os.path.abspath(script14_af), local_af)
    print(f'cached {len(os.listdir(local_af))} AF PDBs')

# Mirror to Drive so future sessions don't re-download
!mkdir -p {drive_af}
!cp -n {local_af}/*.pdb {drive_af}/ 2>/dev/null
print(f"Drive mirror: {len(os.listdir(drive_af))} PDBs")

## 4. Cache T2837 SaProt embeddings (~5 min on T4)

Same mutation-aware feature shape as ESM2 (`h_site + h_window + wt_oh + mut_oh`, 2600-d), so all downstream scripts (06, 18, 19) work unchanged.

**Crucial:** we use the *same split* as the ESM2 T2837 cache to keep an apples-to-apples comparison.  Script 20 inherits `split` from the metadata CSV when present.

In [ ]:
!python scripts/20_cache_embeddings_saprot.py \
    --metadata-csv cache/t2837_metadata.csv \
    --pdb-dir      af_pdbs \
    --pdb-pattern  'AF-{uniprot_id}.pdb' \
    --out          cache/t2837_embeddings_saprot.pt \
    --metadata-out cache/t2837_metadata_saprot.csv \
    --device cuda --seed 42

## 5. Cache S669 SaProt embeddings (~3 min on T4)

In [ ]:
!python scripts/20_cache_embeddings_saprot.py \
    --metadata-csv cache/s669_metadata.csv \
    --pdb-dir      data/s669/S669/pdbs \
    --pdb-pattern  '{pdb_code}.pdb' \
    --pdb-pattern-fallback '{pdb_code_lower}.pdb' \
    --val-fraction 0 --test-fraction 1.0 \
    --out          cache/s669_embeddings_saprot.pt \
    --metadata-out cache/s669_metadata_saprot.csv \
    --device cuda --seed 42

## 6. Build SaProt-aligned bio features

The bio features (k=13) are *backbone-independent* by construction (they depend only on AA identity, RSA, and sequence-derived structural proxies — none of which change when the encoder changes).  We re-build them against the new metadata so row counts align with the SaProt cache, but the standardiser comes from the **T2837 SaProt train split** for T2837 and from the same T2837 mu/sd for S669.

In [ ]:
# T2837 SaProt: build standard bio features against the SaProt-aligned metadata
!python scripts/06_build_bio_features.py \
    --metadata-csv cache/t2837_metadata_saprot.csv \
    --embeddings   cache/t2837_embeddings_saprot.pt \
    --out          cache/t2837_bio_features_saprot_extended.pt \
    --include-extended

In [ ]:
# S669 SaProt: inline build standardised with T2837 SaProt train mu/sd
import sys, numpy as np, pandas as pd, torch
sys.path.insert(0, '/content/uapp')
from uapp.data import load_cached_embeddings
from uapp.mutation_features import batch_features_extended

THREE_TO_ONE = {
    'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLU':'E','GLN':'Q',
    'GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F',
    'PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V',
}
to1 = lambda x: THREE_TO_ONE.get(str(x).strip().upper(), str(x).strip().upper()[:1] or 'X')

md = pd.read_csv('cache/s669_metadata_saprot.csv')
md['split'] = md['split'].astype(str).str.lower()
splits, _ = load_cached_embeddings('cache/s669_embeddings_saprot.pt')
nonempty = [s for s, (X, _) in splits.items() if X.shape[0] > 0]
assert len(nonempty) == 1, nonempty
split_key = nonempty[0]
n_cache = splits[split_key][0].shape[0]
g = md[md['split'] == split_key].reset_index(drop=True)
assert len(g) == n_cache, f'mismatch: md {len(g)} vs cache {n_cache}'

raw = batch_features_extended(
    [to1(x) for x in g['wtAA']], [to1(x) for x in g['mutAA']],
    g['rel_rsa'].astype(float).to_numpy(),
    sequences=g['sequence'].astype(str).tolist(),
    mut_indices=g['mut_idx'].astype(int).to_numpy(),
    include_indicators=False,
)
t_bio = torch.load('cache/t2837_bio_features_saprot_extended.pt',
                   map_location='cpu', weights_only=False)
mu = np.asarray(t_bio['meta']['mu'], dtype=np.float64)
sd = np.asarray(t_bio['meta']['sd'], dtype=np.float64)
sd = np.where(sd < 1e-6, 1.0, sd)
standardised = ((raw - mu) / sd).astype(np.float32)

payload = {k: {'feats': torch.zeros(0, len(mu))} for k in ('train', 'val', 'test')}
payload[split_key] = {'feats': torch.from_numpy(standardised)}
payload['meta'] = {**t_bio['meta'],
                   'source_metadata':   'cache/s669_metadata_saprot.csv',
                   'source_embeddings': 'cache/s669_embeddings_saprot.pt',
                   'standardiser_from': 'cache/t2837_bio_features_saprot_extended.pt'}
torch.save(payload, 'cache/s669_bio_features_saprot_extended.pt')
print(f'Saved cache/s669_bio_features_saprot_extended.pt  ({standardised.shape})')

## 7. Train D5 ensemble on T2837 SaProt, evaluate on T2837 test + S669

Same script 18, just different cache files.

In [ ]:
!python scripts/18_evaluate_on_s669.py \
    --t2837-emb cache/t2837_embeddings_saprot.pt \
    --t2837-bio cache/t2837_bio_features_saprot_extended.pt \
    --s669-emb  cache/s669_embeddings_saprot.pt \
    --s669-bio  cache/s669_bio_features_saprot_extended.pt \
    --out       outputs/saprot_eval_d5 \
    --ablation D5 --members 5 --device cuda

## 8. σ recalibration on S669 (SaProt edition)

Re-fit T on the SaProt predictions — the value is encoder-specific.  Compare to the ESM2 baseline T = 2.76.

In [ ]:
!python scripts/19_recalibrate_sigma_s669.py \
    --predictions outputs/saprot_eval_d5/per_member_predictions_s669.npz \
    --out         outputs/saprot_recalibration

## 9. Save results to Drive + headline comparison

In [ ]:
!cp -r outputs/saprot_eval_d5 outputs/saprot_recalibration /content/drive/MyDrive/uapp_cache/
!cp cache/t2837_embeddings_saprot.pt cache/t2837_metadata_saprot.csv \
    cache/t2837_bio_features_saprot_extended.pt \
    cache/s669_embeddings_saprot.pt cache/s669_metadata_saprot.csv \
    cache/s669_bio_features_saprot_extended.pt /content/drive/MyDrive/uapp_cache/
print('saved to Drive')

In [ ]:
import json, pathlib
saprot = json.loads(pathlib.Path('outputs/saprot_eval_d5/ensemble_summary.json').read_text())
saprot_recal = json.loads(pathlib.Path('outputs/saprot_recalibration/recalibration_summary.json').read_text())

# ESM2 reference (from REPORT.md §11)
esm2_t2837   = {'rmse': 1.50, 'nll': 1.85, 'ice': 0.05, 'spearman': 0.348}
esm2_s669    = {'rmse': 2.83, 'nll': 4.64, 'ice': 0.31, 'spearman': 0.434}
esm2_temp_T  = 2.76

saprot_t2837 = saprot['t2837_test']['ensemble']
saprot_s669  = saprot['s669']['ensemble']
saprot_temp_T = saprot_recal['temperature']

row = lambda r: f"  rmse={r['rmse']:.3f}  nll={r['nll']:.3f}  ice={r['ice']:.3f}  spearman={r['spearman']:.3f}"
print('=' * 78)
print('Encoder comparison — ESM2-650M (§11) vs SaProt (this run)')
print('=' * 78)
print('T2837 test (n=170)')
print('  ESM2-650M:', f"  rmse={esm2_t2837['rmse']:.3f}  nll={esm2_t2837['nll']:.3f}  ice={esm2_t2837['ice']:.3f}  spearman={esm2_t2837['spearman']:.3f}")
print('  SaProt:   ', row(saprot_t2837))
print()
print('S669 (n=617)')
print('  ESM2-650M:', f"  rmse={esm2_s669['rmse']:.3f}  nll={esm2_s669['nll']:.3f}  ice={esm2_s669['ice']:.3f}  spearman={esm2_s669['spearman']:.3f}")
print('  SaProt:   ', row(saprot_s669))
print()
print(f'σ recalibration on S669:  ESM2 T = {esm2_temp_T:.2f}    SaProt T = {saprot_temp_T:.2f}')
print('=' * 78)

## Decision rule

| Outcome | What it means |
|---|---|
| SaProt T2837 RMSE ≤ 1.50 **and** Spearman ≥ 0.348, **and** SaProt S669 RMSE significantly < 2.83 | **Strict win.** Add SaProt as the production encoder; keep D5 head + recalibration recipe. |
| SaProt T2837 unchanged but S669 RMSE unchanged | Structure-awareness doesn't fix the cross-dataset μ gap on its own. Document; try ESM-3 / ESM-IF next. |
| SaProt T2837 *worse* than ESM2 | Encoder swap broke in-distribution performance. Negative result; the gain on S669 (if any) is not free. |
| Any SaProt run with Spearman « 0.348 | The σ-ranking property didn't survive. SaProt's combined AA+3Di tokens may have changed what the σ branch sees. |